In [1]:
%matplotlib inline

import json
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

In [2]:
JSON_PATH = "/Users/pri/Downloads/split_vehicle_files/866082077547702.json"
OUTPUT_DIR = "six_sense/outputs"

In [3]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

NOMINAL_V = 51.2
V_LOW_FLAG = 40.0
V_HIGH_FLAG = 56.0
V_CRITICAL = 70.0

CURRENT_RAW_THRESHOLD = 6000

In [4]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Raw JSON records loaded: {len(raw):,}")

records = []

for row in raw:
    src = row.get("_source", {}).copy()

    src["_id"] = row.get("_id")
    src["_index"] = row.get("_index")

    if "sort" in row:
        src["_sort"] = row["sort"][0] if isinstance(row["sort"], list) and row["sort"] else None

    records.append(src)

df = pd.DataFrame(records)

print(f"Flattened rows: {len(df):,}")
print(f"Columns found: {len(df.columns)}")

Raw JSON records loaded: 1,691,178
Flattened rows: 1,691,178
Columns found: 27


In [5]:
df.columns

Index(['_class', 'id', 'created', 'modified', 'uid', 'timestamp', 'acc_x',
       'acc_y', 'acc_z', 'total_volt', 'total_amp', 'state_of_charge',
       'state_of_health', 'max_battery_temp', 'min_battery_temp',
       'gen_battery_temp', 'max_cell_volt', 'min_cell_volt', 'motor_rpm',
       'total_vehicle_dist', 'vehicle_status', 'speed', 'remain_range',
       'org_code', '_id', '_index', '_sort'],
      dtype='object')

In [6]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values("timestamp").reset_index(drop=True)

numeric_cols = [
    "acc_x", "acc_y", "acc_z",
    "total_volt", "total_amp",
    "state_of_charge", "state_of_health",
    "max_battery_temp", "min_battery_temp",
    "max_cell_volt", "min_cell_volt",
    "motor_rpm", "total_vehicle_dist",
    "vehicle_status", "speed", "remain_range"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


df["total_amp_raw"] = df["total_amp"]

In [7]:
# What is the time range of the dataframe?
time_min = df["timestamp"].min()
time_max = df["timestamp"].max()
duration = time_max - time_min

# Extract total days, hours
total_days = duration.days
total_hours = duration.total_seconds() // 3600

print(f"Data time range: {time_min} to {time_max}")
print(f"Duration: {total_days} days ({int(total_hours)} hours)")

Data time range: 2025-10-27 01:17:50 to 2026-03-03 13:55:30
Duration: 127 days (3060 hours)


In [8]:
df.columns

Index(['_class', 'id', 'created', 'modified', 'uid', 'timestamp', 'acc_x',
       'acc_y', 'acc_z', 'total_volt', 'total_amp', 'state_of_charge',
       'state_of_health', 'max_battery_temp', 'min_battery_temp',
       'gen_battery_temp', 'max_cell_volt', 'min_cell_volt', 'motor_rpm',
       'total_vehicle_dist', 'vehicle_status', 'speed', 'remain_range',
       'org_code', '_id', '_index', '_sort', 'total_amp_raw'],
      dtype='object')

In [9]:
import pandas as pd

# Specify only the columns of interest
cols_of_interest = [
    "timestamp",
    "acc_x",
    "acc_y",
    "acc_z",
    "total_volt",
    "total_amp",
    "state_of_charge",
    "state_of_health",
    "max_battery_temp",
    "min_battery_temp",
    "gen_battery_temp",
    "max_cell_volt",
    "min_cell_volt",
    "motor_rpm",
    "total_vehicle_dist",
    "vehicle_status",
    "speed",
    "remain_range"
]

# Filter df for only the listed columns that actually exist (to avoid errors if some are missing)
present_cols = [col for col in cols_of_interest if col in df.columns]

null_counts = df[present_cols].isnull().sum()
zero_counts = (df[present_cols] == 0).sum(numeric_only=True)

summary = pd.DataFrame({
    "null_count": null_counts,
    "zero_count": zero_counts
}).fillna(0).astype(int)

print("Null and Zero Value Summary (selected columns):")
print(summary)

Null and Zero Value Summary (selected columns):
                    null_count  zero_count
timestamp                    0           0
acc_x                        0           0
acc_y                        0        3455
acc_z                        0       24107
total_volt                   0         370
total_amp                    0      561677
state_of_charge              0           0
state_of_health              0     1691178
max_battery_temp             0     1691178
min_battery_temp        520900     1170278
gen_battery_temp        520900     1170278
max_cell_volt                0      520910
min_cell_volt                0          10
motor_rpm                    0     1159513
total_vehicle_dist           0         100
vehicle_status               0     1691178
speed                        0      876536
remain_range                 0           6


In [10]:
# Calculate the elapsed time (delta t) between consecutive records
df["delta_t"] = df["timestamp"].diff().dt.total_seconds()
descr = df["delta_t"].describe()
median = df["delta_t"].median()
print(descr)
print(f"Median: {median}")

count    1.691177e+06
mean     6.515143e+00
std      1.985534e+02
min      0.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.283310e+05
Name: delta_t, dtype: float64
Median: 4.0


In [11]:
def print_largest_gap_summary(df):
    """
    Find and print the largest gap in 'delta_t' (in seconds) and its conversions in minutes, hours, and days.
    """
    max_gap_seconds = df["delta_t"].max()
    print(f"Largest gap between records: {max_gap_seconds:.1f} seconds")

    # Convert to minutes, hours, and days
    if pd.notnull(max_gap_seconds):
        max_gap_minutes = max_gap_seconds / 60
        max_gap_hours = max_gap_seconds / 3600
        max_gap_days = max_gap_seconds / 86400
        print(f"Largest gap in minutes: {max_gap_minutes:.2f} minutes")
        print(f"Largest gap in hours: {max_gap_hours:.2f} hours")
        print(f"Largest gap in days: {max_gap_days:.4f} days")
    else:
        print("No valid 'delta_t' values found.")

# Example usage:
print_largest_gap_summary(df)

Largest gap between records: 128331.0 seconds
Largest gap in minutes: 2138.85 minutes
Largest gap in hours: 35.65 hours
Largest gap in days: 1.4853 days


In [12]:
def show_soc_and_voltage_around_largest_gap(df):
    """
    Examine and print the SOC and voltage just before and after the largest gap in delta_t.
    Also print the change in these values across the gap.
    """
    # Locate the index of the largest gap
    idx_max_gap = df["delta_t"].idxmax()

    if pd.notnull(idx_max_gap) and 1 <= idx_max_gap < len(df):
        # Row before the gap
        row_before_gap = df.iloc[idx_max_gap - 1]
        # Row after the gap
        row_after_gap = df.iloc[idx_max_gap]

        print("\nValues surrounding the largest time gap:")
        print(f"Time BEFORE gap: {row_before_gap['timestamp']}")
        print(f"    State of Charge BEFORE: {row_before_gap.get('state_of_charge', 'N/A')}")
        print(f"    Voltage BEFORE: {row_before_gap.get('total_volt', 'N/A')}")
        print(f"Time AFTER gap: {row_after_gap['timestamp']}")
        print(f"    State of Charge AFTER: {row_after_gap.get('state_of_charge', 'N/A')}")
        print(f"    Voltage AFTER: {row_after_gap.get('total_volt', 'N/A')}")

        delta_soc = row_after_gap.get('state_of_charge', None) - row_before_gap.get('state_of_charge', None) if (
            pd.notnull(row_after_gap.get('state_of_charge', None)) and pd.notnull(row_before_gap.get('state_of_charge', None))
        ) else None

        delta_volt = row_after_gap.get('total_volt', None) - row_before_gap.get('total_volt', None) if (
            pd.notnull(row_after_gap.get('total_volt', None)) and pd.notnull(row_before_gap.get('total_volt', None))
        ) else None

        print(f"\nChange in State of Charge (SOC) across gap: {delta_soc}")
        print(f"Change in Voltage (total_volt) across gap: {delta_volt}")
    else:
        print("Could not determine rows before and after the largest gap.")

# Example usage:
show_soc_and_voltage_around_largest_gap(df)


Values surrounding the largest time gap:
Time BEFORE gap: 2026-01-28 22:41:32
    State of Charge BEFORE: 37.0
    Voltage BEFORE: 49.29
Time AFTER gap: 2026-01-30 10:20:23
    State of Charge AFTER: 35.0
    Voltage AFTER: 49.27

Change in State of Charge (SOC) across gap: -2.0
Change in Voltage (total_volt) across gap: -0.01999999999999602


In [13]:
def summarize_signal_ranges(df):
    """
    Summarizes observed ranges and checks against expected ranges for vehicle telemetry signals.
    For 'total_vehicle_dist' (odometer), also returns (and prints) a DataFrame of non-monotonic instances,
    showing before/after timestamps, values, the delta, the time gap, and whether a time hole (>threshold) is present.

    Additionally, for odometer decreases, checks if decrease coincided with a time gap ('hole'),
    and suggests to user that with large time holes + non-monotonic odometer,
    odometer reset, system sync, or data ingest issues may be at play.

    Parameters:
        df (pd.DataFrame): Dataframe with signal columns

    Prints:
        Range summary as a formatted DataFrame string.
        (and for odometer, table of decreases and time hole notes if any)
    Returns:
        df_odometer_viol: DataFrame of odometer (total_vehicle_dist) non-monotonic instances, or None if monotonic
    """
    # Define expected ranges for each signal
    expected_ranges = {
        "total_volt":         (40, 56),          # V
        "total_amp":          (-200, 200),          # A, ~120
        "state_of_charge":    (0, 100),          # %
        "state_of_health":    (0, 100),         # %
        "max_battery_temp":   (-20, 60),          # °C
        "min_battery_temp":   (-20, 60),          # °C
        "gen_battery_temp":   (-20, 60),          # °C
        "max_cell_volt":      (3.0, 3.65),       # V
        "min_cell_volt":      (3.0, 3.65),       # V
        "motor_rpm":          (0, 6000),         # ~6000
        "speed":              (0, 100),          # km/h
        "total_vehicle_dist": ("monotonic",),    # Special case
        "remain_range":       ("soc_prop",),     # Special case
    }

    expected_strings = {
        "total_volt":         "44 – 56 V",
        "total_amp":          "0 – ~120 A",
        "state_of_charge":    "0 – 100 %",
        "state_of_health":    "70 – 100 %",
        "max_battery_temp":   "10 – 60 °C",
        "min_battery_temp":   "10 – 60 °C",
        "gen_battery_temp":   "10 – 60 °C",
        "max_cell_volt":      "3.0 – 3.65 V",
        "min_cell_volt":      "3.0 – 3.65 V",
        "motor_rpm":          "0 – ~6,000",
        "speed":              "0 – 100 km/h",
        "total_vehicle_dist": "Monotonic ↑",
        "remain_range":       "Proportional to SOC",
    }

    signal_order = [
        "total_volt",
        "total_amp",
        "state_of_charge",
        "state_of_health",
        "max_battery_temp",
        "min_battery_temp",
        "gen_battery_temp",
        "max_cell_volt",
        "min_cell_volt",
        "motor_rpm",
        "speed",
        "total_vehicle_dist",
        "remain_range",
    ]

    def pretty_number(x):
        if pd.isnull(x):
            return "N/A"
        if isinstance(x, float) and x.is_integer():
            x = int(x)
        if isinstance(x, int):
            return f"{x:,d}"
        if isinstance(x, float):
            return f"{x:.3f}".rstrip('0').rstrip('.') if '.' in f"{x:.3f}" else f"{x:.3f}"
        return str(x)

    # For odometer non-monotonic instances, will collect here
    df_odometer_viol = None
    odometer_hole_threshold = pd.Timedelta(minutes=10)  # Define what we call a "hole"

    rows = []
    odometer_hole_flag = False  # To record if we see any time hole associated

    for sig in signal_order:
        exp_rng = expected_ranges[sig]
        exp_str = expected_strings[sig]
        note = ""
        actual_range = "not found"
        observed_range = "N/A"
        sig_col_present = sig in df.columns and df[sig].notna().sum() > 0
        if not sig_col_present:
            rows.append({
                "Signal": sig,
                "Range Observed": observed_range,
                "Expected": exp_str,
                "Notes": "Not found"
            })
            continue

        col = df[sig].dropna()

        # Compute actual observed range
        if isinstance(exp_rng[0], (int, float)):
            col_min, col_max = col.min(), col.max()
            units = exp_str.split()[-1] if len(exp_str.split()) > 1 else ""
            if units in ["V", "%", "A", "°C", "km/h"]:
                observed_range = f"{pretty_number(col_min)} – {pretty_number(col_max)} {units}"
            elif sig == "motor_rpm":
                observed_range = f"{pretty_number(col_min)} – {pretty_number(col_max)}"
            else:
                observed_range = f"{pretty_number(col_min)} – {pretty_number(col_max)}"
            actual_range = observed_range
        elif exp_rng[0] == "monotonic":
            is_monotonic = df[sig].is_monotonic_increasing
            col_min, col_max = col.min(), col.max()
            observed_range = f"{pretty_number(col_min)} – {pretty_number(col_max)}"
            if is_monotonic:
                actual_range = "Monotonic ↑"
            else:
                diffs = df[sig].diff()
                n_nonmono = (diffs < 0).sum()
                perc_nonmono = 100 * n_nonmono / len(diffs.dropna()) if len(diffs.dropna()) > 0 else 0
                note = f"{n_nonmono} decreases ({perc_nonmono:.2f}%)"
                actual_range = "Not strictly ↑"
                # For odometer, build violations DataFrame for user inspection
                if sig == "total_vehicle_dist":
                    # Find the indices where it goes down
                    nonmono_idx = diffs[diffs < 0].index
                    # For each, grab prev and curr, and check their timestamp gap
                    records = []
                    odom_holes = []
                    for i in nonmono_idx:
                        prev_row = df.loc[i-1] if i-1 in df.index else None
                        curr_row = df.loc[i]
                        if prev_row is not None and curr_row is not None:
                            prev_val = prev_row[sig]
                            curr_val = curr_row[sig]
                            delta = curr_val - prev_val
                            # Time before/after
                            t_before = prev_row["timestamp"] if "timestamp" in prev_row else prev_row.name
                            t_after = curr_row["timestamp"] if "timestamp" in curr_row else curr_row.name

                            # Compute time gap (if parseable)
                            try:
                                t_before_ts = pd.to_datetime(t_before)
                                t_after_ts = pd.to_datetime(t_after)
                                time_gap = t_after_ts - t_before_ts
                            except Exception:
                                time_gap = pd.NaT
                            is_hole = pd.notnull(time_gap) and time_gap > odometer_hole_threshold
                            if is_hole:
                                odometer_hole_flag = True
                            rec = {
                                "timestamp_before": t_before,
                                "total_vehicle_dist_before": prev_val,
                                "timestamp_after": t_after,
                                "total_vehicle_dist_after": curr_val,
                                "delta": delta,
                                "time_gap": time_gap,
                                "time_hole": is_hole
                            }
                            records.append(rec)
                            if is_hole:
                                odom_holes.append((t_before, t_after))
                    if len(records) > 0:
                        df_odometer_viol = pd.DataFrame(records)
                        print("\nOdometer (total_vehicle_dist) non-monotonic instances (vehicle distance decreased!):")
                        # Print the dataframe with info about time holes
                        # Show the table with a little formatting
                        pd_display = df_odometer_viol.copy()
                        if 'time_gap' in pd_display.columns:
                            pd_display['time_gap'] = pd_display['time_gap'].apply(lambda x: str(x) if pd.notnull(x) else "N/A")
                        print(pd_display.to_string(index=False))
                        if odometer_hole_flag:
                            print(
                                "\nNOTE: One or more odometer decreases coincided with a large time gap (hole) between readings."
                                " This can suggest possible odometer reset, telematics unit power-cycle, or a gap in data ingest."
                                " Please check raw data for possible resets, system reboots, or timesync issues."
                            )
                        else:
                            print(
                                "\nNone of the odometer decreases coincided with a large time gap. If odometer went backward"
                                " with regular (high-frequency) reporting, this may indicate errant data, telemetry bug,"
                                " or odometer clock slip. Please review signal for sensor/data anomalies."
                            )
                    else:
                        df_odometer_viol = None
        elif exp_rng[0] == "soc_prop":
            observed_range = "N/A"
            actual_range = "Proportional to SOC"

        # Range check
        if isinstance(exp_rng[0], (int, float)):
            lo, hi = exp_rng
            # Use soft upper for ~values
            fudge_hi = hi * 1.05 if "~" in exp_str or sig in ["motor_rpm", "total_amp"] else hi
            outside_lo = (col < lo)
            outside_hi = (col > fudge_hi)
            outside_count = (outside_lo | outside_hi).sum()
            total = len(col)
            if outside_count > 0:
                perc = 100 * outside_count / total if total > 0 else 0
                note = f"{outside_count} ({perc:.2f}%) out of expected range"
        elif sig == "total_vehicle_dist":
            # already handled above
            pass
        elif sig == "remain_range":
            # Not easily checked quantitatively
            pass

        rows.append({
            "Signal": sig,
            "Range Observed": observed_range,
            "Expected": exp_str,
            "Notes": note if note else ""
        })

    summary_df = pd.DataFrame(rows, columns=["Signal", "Range Observed", "Expected", "Notes"])
    print(summary_df.to_string(index=False, header=True, col_space=20))

    return df_odometer_viol

# Example usage:
df_odometer_viol = summarize_signal_ranges(df)


Odometer (total_vehicle_dist) non-monotonic instances (vehicle distance decreased!):
   timestamp_before  total_vehicle_dist_before     timestamp_after  total_vehicle_dist_after     delta        time_gap  time_hole
2025-10-27 21:43:25                   108570.0 2025-10-27 21:55:00                       0.0 -108570.0 0 days 00:11:35       True
2025-11-10 14:36:17                   110686.0 2025-11-10 14:36:21                  110681.0      -5.0 0 days 00:00:04      False
2025-11-10 15:09:14                   110707.0 2025-11-10 15:09:16                  110704.0      -3.0 0 days 00:00:02      False
2025-11-10 15:22:26                   110711.0 2025-11-10 15:22:28                  110710.0      -1.0 0 days 00:00:02      False
2025-11-10 15:41:31                   110715.0 2025-11-10 15:41:33                  110711.0      -4.0 0 days 00:00:02      False
2025-11-10 15:41:58                   110715.0 2025-11-10 15:42:02                  110711.0      -4.0 0 days 00:00:04      False
2025

In [14]:
total_amp_min = df["total_amp"].min()
total_amp_max = df["total_amp"].max()
print(f"Observed range of total_amp: min={total_amp_min}, max={total_amp_max}")
if total_amp_min < 0:
    print("total_amp includes both negative and positive values.")
else:
    print("total_amp is only positive (or zero).")

Observed range of total_amp: min=0.0, max=6553.1
total_amp is only positive (or zero).


In [ ]:
df_clean = df[df["total_amp_raw"].abs() <= CURRENT_RAW_THRESHOLD].copy()

removed_rows = len(df) - len(df_clean)
num_negative_current = (df["total_amp_raw"] < 0).sum()

print("\nCURRENT FILTER")
print("-" * 70)
print(f"Threshold                     : abs(current_raw) <= {CURRENT_RAW_THRESHOLD}")
print(f"Rows removed as suspicious    : {removed_rows:,}")
print(f"Rows kept in clean dataset    : {len(df_clean):,}")
print(f"Rows with negative current    : {num_negative_current:,}")

In [ ]:
def plot_current_and_soc_by_day(df):
    """
    Visualize current and SOC for each day as separate plots (displayed only, not saved).

    Parameters:
        df (pd.DataFrame): Dataframe containing at least 'timestamp', 'total_amp', 'state_of_charge' columns.
    """
    import matplotlib.pyplot as plt
    import pandas as pd

    # Get list of unique days in the data
    unique_days = df["timestamp"].dt.normalize().sort_values().unique()

    print(f"Found {len(unique_days)} distinct days in data.")

    for the_day in unique_days:
        # Construct start and end times for the day
        day_start = pd.to_datetime(the_day)
        day_end = day_start + pd.Timedelta(days=1)

        day_mask = (df["timestamp"] >= day_start) & (df["timestamp"] < day_end)
        df_day = df[day_mask]

        if df_day.empty:
            print(f"No data for day {day_start.date()}, skipping.")
            continue

        print(f"Plotting for day: {day_start.date()} (rows: {len(df_day)})")

        fig, ax1 = plt.subplots(figsize=(14, 6))
        color_current = 'tab:red'
        color_soc = 'tab:blue'

        # Plot Current
        ax1.set_xlabel('Time')
        ax1.set_ylabel('Total Current (A)', color=color_current)
        ax1.plot(df_day["timestamp"], df_day["total_amp"], color=color_current, label="Current (A)")
        ax1.tick_params(axis='y', labelcolor=color_current)

        # Second y axis for SOC
        ax2 = ax1.twinx()
        ax2.set_ylabel('State of Charge (%)', color=color_soc)
        ax2.plot(df_day["timestamp"], df_day["state_of_charge"], color=color_soc, label="SOC (%)")
        ax2.tick_params(axis='y', labelcolor=color_soc)

        plt.title(f"Current (total_amp) and SOC over Time [day: {day_start.date()}]")
        fig.tight_layout()

        # Add grid and show legend
        ax1.grid(True, which='both', axis='both', linestyle='--', alpha=0.3)
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')

        plt.show()

    # Quick visual guide
    print("If SOC is increasing while current is positive: positive current charges the battery.")
    print("If SOC is increasing while current is negative: negative current charges the battery.")
    print("Examine the chart visually: look for SOC increase periods and check the corresponding current sign.")

# Example usage:
plot_current_and_soc_by_day(df_clean)

In [ ]:
plot_current_and_soc_by_day(df_clean)

In [ ]:
def plot_all_sensor_timeseries(df, time_col="timestamp"):
    """
    Plots all temperature, voltage, and current sensor columns over time from a DataFrame,
    EXCLUDING columns named 'timestamp' and any column containing 'amp_raw' in their name.
    The columns are selected based on whether their name contains 'temp', 'volt', 'amp', or 'curr' (case-insensitive),
    but excludes 'amp_raw' and 'timestamp' columns from being plotted as sensors.
    
    Args:
        df (pd.DataFrame): The DataFrame containing the telemetry data.
        time_col (str): The name of the timestamp column used as the x-axis.
    """
    import matplotlib.pyplot as plt

    # Identify sensor columns, but exclude 'timestamp' and 'amp_raw' columns
    temp_cols = [col for col in df.columns if "temp" in col.lower()
                 and col.lower() != time_col.lower()
                 and "amp_raw" not in col.lower()]
    volt_cols = [col for col in df.columns if "volt" in col.lower()
                 and col.lower() != time_col.lower()
                 and "amp_raw" not in col.lower()]
    curr_cols = [col for col in df.columns if
                 (("amp" in col.lower() or "curr" in col.lower())
                  and "amp_raw" not in col.lower()
                  and col.lower() != time_col.lower())]

    n_temp = len(temp_cols)
    n_volt = len(volt_cols)
    n_curr = len(curr_cols)
    
    # Plot Temperature Sensors
    if n_temp > 0:
        plt.figure(figsize=(16, 4 * n_temp))
        for i, col in enumerate(temp_cols, 1):
            plt.subplot(n_temp, 1, i)
            plt.plot(df[time_col], df[col], label=col)
            plt.xlabel("Timestamp")
            plt.ylabel(col)
            plt.title(f"{col} over Time")
            plt.legend()
            plt.tight_layout()
        plt.show()
    else:
        print("No temperature sensor columns found.")

    # Plot Voltage Sensors
    if n_volt > 0:
        plt.figure(figsize=(16, 4 * n_volt))
        for i, col in enumerate(volt_cols, 1):
            plt.subplot(n_volt, 1, i)
            plt.plot(df[time_col], df[col], label=col, color="tab:blue")
            plt.xlabel("Timestamp")
            plt.ylabel("Voltage (V)")
            plt.title(f"{col} over Time")
            plt.legend()
            plt.tight_layout()
        plt.show()
    else:
        print("No voltage sensor columns found.")

    # Plot Current Sensors
    if n_curr > 0:
        plt.figure(figsize=(16, 4 * n_curr))
        for i, col in enumerate(curr_cols, 1):
            plt.subplot(n_curr, 1, i)
            plt.plot(df[time_col], df[col], label=col, color="tab:orange")
            plt.xlabel("Timestamp")
            plt.ylabel("Current (A)")
            plt.title(f"{col} over Time")
            plt.legend()
            plt.tight_layout()
        plt.show()
    else:
        print("No current sensor columns found.")

# Example usage:
plot_all_sensor_timeseries(df)

In [ ]:
# Plot State of Charge (SOC) and Cell Voltages over time

plt.figure(figsize=(16, 4))

plt.plot(df_clean["timestamp"], df_clean["state_of_charge"], label="State of Charge (%)", color="tab:green")
plt.xlabel("Timestamp")
plt.ylabel("State of Charge (%)")
plt.title("State of Charge (SOC) over Time")
plt.legend()
plt.tight_layout()
plt.show()

# Plot min and max cell voltages over time
plt.figure(figsize=(16, 4))

if "min_cell_volt" in df_clean.columns and "max_cell_volt" in df_clean.columns:
    plt.plot(df_clean["timestamp"], df_clean["min_cell_volt"], label="Min Cell Voltage (V)", color="tab:blue")
    plt.plot(df_clean["timestamp"], df_clean["max_cell_volt"], label="Max Cell Voltage (V)", color="tab:red")
    plt.xlabel("Timestamp")
    plt.ylabel("Cell Voltage (V)")
    plt.title("Min and Max Cell Voltages over Time")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("min_cell_volt or max_cell_volt columns not found in dataframe.")

In [ ]:
deadband = 0.5

# SOC change vs previous row (signed). Only increases indicate net charging into the pack.
soc_delta = df_clean["state_of_charge"].diff()

# Determine if charging current is positive or negative, based on SOC increase events
charging_events = df_clean[(soc_delta > 0.1) & soc_delta.notnull()]

if not charging_events.empty:
    sign_during_increase = charging_events["total_amp"].apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0)).median()
    
    if sign_during_increase > 0:
        charging_current_sign = "positive"
    elif sign_during_increase < 0:
        charging_current_sign = "negative"
    else:
        charging_current_sign = "zero or unclear"
else:
    charging_current_sign = "no charging events detected"

print(f"Charging current appears to be: {charging_current_sign}")

# Now, use the determined charging current sign to select the right mask logic
if charging_current_sign == "positive":
    charging_mask = df_clean["total_amp"] > deadband
    discharging_mask = df_clean["total_amp"] < -deadband
elif charging_current_sign == "negative":
    charging_mask = df_clean["total_amp"] < -deadband
    discharging_mask = df_clean["total_amp"] > deadband
else:
    # fallback to both directions as ambiguous
    charging_mask = abs(df_clean["total_amp"]) > deadband
    discharging_mask = abs(df_clean["total_amp"]) > deadband

resting_mask = (~charging_mask) & (~discharging_mask)

num_charging = charging_mask.sum()
num_discharging = discharging_mask.sum()
num_resting = resting_mask.sum()

print(f"With deadband {deadband}:")
print(f"Rows charging    : {num_charging:,}")
print(f"Rows discharging : {num_discharging:,}")
print(f"Rows resting     : {num_resting:,}")

In [ ]:
# Visualize current and SOC for each day as separate plots

import matplotlib.pyplot as plt

# Get list of unique days in the data
unique_days = df_clean["timestamp"].dt.normalize().sort_values().unique()

print(f"Found {len(unique_days)} distinct days in data.")

for the_day in unique_days:
    # Construct start and end times for the day
    day_start = pd.to_datetime(the_day)
    day_end = day_start + pd.Timedelta(days=1)

    day_mask = (df_clean["timestamp"] >= day_start) & (df_clean["timestamp"] < day_end)
    df_day = df_clean[day_mask]

    if df_day.empty:
        print(f"No data for day {day_start.date()}, skipping.")
        continue

    print(f"Plotting for day: {day_start.date()} (rows: {len(df_day)})")

    fig, ax1 = plt.subplots(figsize=(14, 6))
    color_current = 'tab:red'
    color_soc = 'tab:blue'

    # Plot Current
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Total Current (A)', color=color_current)
    ax1.plot(df_day["timestamp"], df_day["total_amp"], color=color_current, label="Current (A)")
    ax1.tick_params(axis='y', labelcolor=color_current)

    # Second y axis for SOC
    ax2 = ax1.twinx()
    ax2.set_ylabel('State of Charge (%)', color=color_soc)
    ax2.plot(df_day["timestamp"], df_day["state_of_charge"], color=color_soc, label="SOC (%)")
    ax2.tick_params(axis='y', labelcolor=color_soc)

    plt.title(f"Current (total_amp) and SOC over Time [day: {day_start.date()}]")
    fig.tight_layout()

    # Add grid and show legend
    ax1.grid(True, which='both', axis='both', linestyle='--', alpha=0.3)
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')

    out_png = os.path.join(OUTPUT_DIR, f"current_soc_over_time_{day_start.date()}.png")
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    print(f"Saved figure to: {out_png}")

    plt.show()

# Quick visual guide
print("If SOC is increasing while current is positive: positive current charges the battery.")
print("If SOC is increasing while current is negative: negative current charges the battery.")
print("Examine the chart visually: look for SOC increase periods and check the corresponding current sign.")

In [ ]:
print("Assigning is_moving, is_charging, is_idle columns to df_clean...")
df_clean["is_moving"] = df_clean["speed"] > 1
df_clean["is_charging"] = (df_clean["total_amp"] < -5) & (~df_clean["is_moving"])
df_clean["is_idle"] = (~df_clean["is_moving"]) & (~df_clean["is_charging"])
print("Done. Value counts for each state:")
print("is_moving:", df_clean["is_moving"].sum())
print("is_charging:", df_clean["is_charging"].sum())
print("is_idle:", df_clean["is_idle"].sum())